# Module 07 · Dao động và IC 555 (Floyd chương 16)

In [1]:
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
%matplotlib inline

## 1. Công thức tần số các mạch RC và LC

In [2]:
print(f"Wien (6.2k, 0.02uF): {1/(2*np.pi*6.2e3*0.02e-6):.0f} Hz")
print(f"Phase-shift (4.7k, 0.022uF): {1/(2*np.pi*np.sqrt(6)*4.7e3*0.022e-6):.0f} Hz, Rf = {29*4.7:.1f} kOhm")
c1, c2, L = 470e-12, 47e-12, 1e-3; ct = c1*c2/(c1+c2)
print(f"Colpitts: CT = {ct*1e12:.1f} pF, f = {1/(2*np.pi*np.sqrt(L*ct))/1e3:.0f} kHz, Av = {c1/c2:.0f}")

Wien (6.2k, 0.02uF): 1284 Hz
Phase-shift (4.7k, 0.022uF): 628 Hz, Rf = 136.3 kOhm
Colpitts: CT = 42.7 pF, f = 770 kHz, Av = 10


**Nhận xét:** 1284 Hz, 628 Hz (đúng sách) và Rf = 136,3 kΩ; mạch Colpitts có Av = 10 (đúng sách).

## 2. Khởi động và biên độ của dao động (mô hình độ lợi vòng)

In [3]:
# amplitude grows by loop gain each cycle until the limiter saturates it (|v| <= 1)
for g in (0.9, 1.0, 1.05, 1.2):
    v = 1e-3; n = 0
    while abs(v) < 1 and n < 1000: v *= g; n += 1
    print(f"loop gain {g:.2f}: {"dies out" if g < 1 else ("stays at 1 mV" if g == 1 else f"reaches 1 V after {n} cycles")}")

loop gain 0.90: dies out
loop gain 1.00: stays at 1 mV
loop gain 1.05: reaches 1 V after 142 cycles
loop gain 1.20: reaches 1 V after 38 cycles


**Nhận xét:** độ lợi vòng nhỏ hơn 1 làm dao động tắt, bằng 1 giữ nguyên biên độ, lớn hơn 1 làm biên độ tăng dần đến khi bị giới hạn — đúng tinh thần Floyd §16–2.

## 3. 555 phi ổn: công thức và mô phỏng tụ nạp/xả

In [4]:
R1, R2, C, Vcc = 1e3, 3.3e3, 0.047e-6, 5.5
f_formula = 1.44/((R1+2*R2)*C); duty = (R1+R2)/(R1+2*R2)
print(f"formula: f = {f_formula:.0f} Hz, duty = {duty*100:.1f} %")
tH = np.log(2)*(R1+R2)*C; tL = np.log(2)*R2*C; print(f"ideal exponentials: tH = {tH*1e6:.1f} us, tL = {tL*1e6:.1f} us -> f = {1/(tH+tL):.0f} Hz")
dt = 1e-8; v = Vcc/3; t = 0; up = True; edges = []
while len(edges) < 8 and t < 5e-3:
    if up:
        v += (Vcc - v)/((R1+R2)*C)*dt
        if v >= 2*Vcc/3: up = False; edges.append(t)
    else:
        v += (0 - v)/(R2*C)*dt
        if v <= Vcc/3: up = True; edges.append(t)
    t += dt
T = (edges[-1]-edges[1])/3
print(f"simulated period = {T*1e6:.1f} us -> f = {1/T:.0f} Hz")

formula: f = 4031 Hz, duty = 56.6 %
ideal exponentials: tH = 140.1 us, tL = 107.5 us -> f = 4039 Hz
simulated period = 247.6 us -> f = 4039 Hz


**Nhận xét:** công thức Floyd cho khoảng 4,03 kHz; mô hình hàm mũ lý tưởng (ln 2 = 0,693 thay cho 0,7) cho tần số rất gần đó, và mô phỏng số khớp với mô hình hàm mũ. Sai khác nhỏ giữa 1,44 và 1/(2·0,693) chỉ là do Floyd làm tròn 0,7.

## 4. Chu kỳ làm việc và thiết kế Cext

In [5]:
print("Cext for 25 kHz:", round(1.44/((1e3+2*3.3e3)*25e3)*1e6, 4), "uF")
print("R2 for 75 % duty with R1 = 3.3k:", 3.3/2, "kOhm")
for r1, r2 in [(1, 1), (1, 10), (10, 1)]: print(f"R1={r1}k R2={r2}k duty = {(r1+r2)/(r1+2*r2)*100:.1f} %")

Cext for 25 kHz: 0.0076 uF
R2 for 75 % duty with R1 = 3.3k: 1.65 kOhm
R1=1k R2=1k duty = 66.7 %
R1=1k R2=10k duty = 52.4 %
R1=10k R2=1k duty = 91.7 %


**Nhận xét:** Cext = 0,0076 μF (đúng sách); R2 = 1,65 kΩ cho 75 %; khi R2 ≪ R1 chu kỳ tiến tới 100 %, khi R2 ≫ R1 tiến tới 50 % — không bao giờ dưới 50 % nếu không có điốt.